# 用 Python 把對話拆成可計算結構

Status: in_use  
Prereq: Python 基礎語法、input()/split()、for 迴圈  
Can-do checklist:  
- [ ] 能把敘述型資料拆成可計算的結構（list/dict）
- [ ] 能設計欄位/狀態表示法並更新
- [ ] 能把流程包成函式並測試

目標：訓練「文字 → 變數 → 規則 → 計算」的思維。

## 學習目標
- 抽取關鍵數字與欄位
- 把敘述改成條件與計算
- 用程式比較不同情境結果

## 知識點
1. 成本與平均成本
2. 支撐/阻力（訂單簿深度）
3. 風險量化（最大回撤）
4. 行為偏誤的規則化

## 1. 文字 → 結構化資料
用縮短片段示範數字抽取與分類。

In [ ]:
import re

raw_text = (
    "Holding: BTC cost 92,722; ADA cost 0.36."
    "Support: BTC 76,500; extreme 72,000."
    "Cash: USDT 94."
    "ADA might rebound to 0.33; consider partial swap."
)

# 1) Extract numbers with commas or decimals
nums = re.findall(r"\d[\d,]*\.?\d*", raw_text)
nums = [float(n.replace(',', '')) for n in nums]
nums

抽數字只是第一步，接著要定義規則把數字歸類。

In [ ]:
# 2) Simple keyword tagging (demo)
lines = [line.strip() for line in raw_text.strip().splitlines() if line.strip()]

items = []
for line in lines:
    if 'cost' in line:
        items.append(('cost', line))
    elif 'Support' in line:
        items.append(('support', line))
    elif 'Cash' in line:
        items.append(('cash', line))
    elif 'rebound' in line:
        items.append(('target', line))

items

分類很粗糙，但重點是「規則設計」。接著把數字取出。

In [ ]:
def extract_numbers(text):
    return [float(n.replace(',', '')) for n in re.findall(r"\d[\d,]*\.?\d*", text)]

structured = []
for tag, line in items:
    structured.append({
        'type': tag,
        'raw': line,
        'numbers': extract_numbers(line)
    })

structured

## 2. 平均成本（純數學）
多筆買入後的平均成本計算。

In [ ]:
# avg cost after multiple buys

def avg_cost_after_buys(current_qty, current_avg, planned_buys):
    # current_qty: current quantity
    # current_avg: current average cost
    # planned_buys: list of (price, usdt_amount)
    current_cost = current_qty * current_avg
    new_qty = current_qty
    new_cost = current_cost

    for price, usdt in planned_buys:
        qty = usdt / price
        new_qty += qty
        new_cost += usdt

    new_avg = new_cost / new_qty if new_qty else 0
    return new_qty, new_avg

# Example (math only)
current_qty = 0.001  # BTC
current_avg = 92722
planned = [(76500, 25), (72000, 25)]

new_qty, new_avg = avg_cost_after_buys(current_qty, current_avg, planned)
new_qty, new_avg

練習：改成輸入多條補倉路徑，輸出各自的平均成本序列。

## 3. 訂單簿深度（支撐/阻力）
用假資料計算累積深度。

In [ ]:
# Fake bids (price, size) - price high to low
bids = [
    (79000, 5.2),
    (78800, 10.5),
    (78500, 30.0),
    (78200, 12.0),
    (78000, 8.0),
    (77500, 40.0),
]

# Fake asks (price, size) - price low to high
asks = [
    (79100, 4.0),
    (79300, 6.5),
    (79500, 18.0),
    (79800, 9.0),
    (80000, 25.0),
]

# Cumulative depth
cum_bids = []
run = 0
for price, size in bids:
    run += size
    cum_bids.append((price, size, run))

cum_asks = []
run = 0
for price, size in asks:
    run += size
    cum_asks.append((price, size, run))

cum_bids, cum_asks

### 簡易「牆」檢測
找出明顯大於平均的掛單。

In [ ]:
import statistics as stats

def detect_walls(levels):
    sizes = [s for _, s in levels]
    threshold = stats.mean(sizes) + stats.pstdev(sizes)
    return [(p, s) for p, s in levels if s >= threshold]

bid_walls = detect_walls(bids)
ask_walls = detect_walls(asks)

bid_walls, ask_walls

注意：訂單簿可能有撤單/假單，資料不等於真相。

## 4. 心理偏誤 → 規則化
把情緒變成變數，控制行為。

In [ ]:
# Mood self-rating (0~10), demo only
mood_score = 8  # 0=calm, 10=very anxious

# If emotion too high, switch to observe-only mode
if mood_score >= 7:
    action = 'observe_only'
else:
    action = 'ok_to_decide'

action

重點：情緒可被程式化成條件。

## 5. 最大回撤
衡量策略最大下滑幅度。

In [ ]:
def max_drawdown(prices):
    peak = prices[0]
    max_dd = 0
    for p in prices:
        if p > peak:
            peak = p
        dd = (peak - p) / peak
        max_dd = max(max_dd, dd)
    return max_dd

sample_prices = [100, 105, 98, 92, 110, 108, 95]
max_drawdown(sample_prices)

## 小結
- 把模糊敘述變成可計算結構
- 用程式比較情境結果

## 練習題
1. 自訂 5 行對話，寫規則抽出成本/支撐/現金/目標。
2. 寫函數：輸入多筆買入，輸出每次買入後平均成本。
3. 生成隨機訂單簿，找出最大的 3 堵牆。
4. 設計一條情緒規則，控制是否允許下單。